# Fase 4 — Verificacion de `get_state()`

Ejecuta un episodio completo en modo **headless** (sin ventana pygame)
y grafica como evolucionan las 14 variables del vector de estado.

**Accion fija:** siempre acelerar `(1.0, 0.0)` — comportamiento determinista y reproducible.

In [ ]:
import os
import sys

# SDL_VIDEODRIVER debe definirse ANTES del import de pygame.
# SDL lee la variable al cargar la libreria, no al llamar pygame.init().
# 'dummy' = driver ficticio: no abre ventana, no necesita GPU ni display.
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')

# El notebook vive en experiments/; el proyecto raiz esta un nivel arriba.
sys.path.insert(0, os.path.abspath('..'))

import pygame
pygame.init()

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from game.environment import Environment

print(f'pygame {pygame.version.ver}  -- imports correctos')

In [ ]:
DT        = 1 / 60      # paso de tiempo en segundos (identico a main.py)
MAX_STEPS = 1200        # techo de 20 s a 60 fps
ACTION    = (1.0, 0.0)  # siempre acelerar, nunca frenar

env = Environment(seed=42)
env.reset()

# Nombres de las 14 entradas — el orden debe coincidir con get_state()
LABELS = [
    'vx',               # 0  velocidad horizontal chasis
    'vy',               # 1  velocidad vertical chasis
    'angulo',           # 2  inclinacion del chasis (rad / pi)
    'omega',            # 3  velocidad angular chasis
    'rueda trasera',    # 4  contacto trasero (0 o 1)
    'rueda delantera',  # 5  contacto delantero (0 o 1)
    'look +30px',       # 6
    'look +80px',       # 7
    'look +150px',      # 8
    'look +250px',      # 9
    'pendiente',        # 10 slope_at(chassis_x)
    'dx moneda',        # 11 horizontal a moneda mas cercana
    'dy moneda',        # 12 vertical a moneda mas cercana
    'tiempo norm.',     # 13 time_left / MAX_TIME
]

# Bucle de simulacion: guarda el vector de estado de cada frame
history   = []
rewards   = []
scores    = []
distances = []

for step_i in range(MAX_STEPS):
    state, reward, done, info = env.step(ACTION, DT)
    history.append(state)
    rewards.append(reward)
    scores.append(info['score'])
    distances.append(info['distance'])
    if done:
        print(f'Episodio termino: frame {step_i}  ({step_i / 60:.1f} s)')
        break
else:
    print(f'Techo alcanzado: {MAX_STEPS} frames ({MAX_STEPS / 60:.0f} s)')

# np.array sobre lista de listas genera shape (n_frames, 14)
# Acceder a una columna: states[:, i] da todos los frames de la variable i
states = np.array(history)
t      = np.arange(len(states)) / 60   # eje de tiempo en segundos

print(f'Frames: {len(states)}   Distancia: {distances[-1]:.0f} px   Score: {scores[-1]}')

In [ ]:
# Agrupamos señales con rango similar para que las escalas sean comparables
GROUPS = [
    ('Velocidades',       [0, 1],       ['vx', 'vy']),
    ('Rotacion',          [2, 3],       ['angulo (/pi)', 'omega (/10 rad/s)']),
    ('Contacto ruedas',   [4, 5],       ['rueda trasera', 'rueda delantera']),
    ('Lookahead terreno', [6, 7, 8, 9], ['+30px', '+80px', '+150px', '+250px']),
    ('Pendiente',         [10],         ['pendiente']),
    ('Vector moneda',     [11, 12],     ['dx', 'dy']),
    ('Tiempo restante',   [13],         ['tiempo norm.']),
]

n_panels = len(GROUPS) + 1   # +1 para el panel de metricas del episodio

fig = plt.figure(figsize=(15, 4 * n_panels))
fig.suptitle(
    'Variables de estado — episodio completo (siempre acelerar, seed=42)',
    fontsize=13, fontweight='bold', y=0.995,
)

# GridSpec controla el espacio vertical entre subplots con hspace.
# hspace=0.55 deja margen para los titulos sin desperdiciar espacio.
gs = gridspec.GridSpec(n_panels, 1, figure=fig, hspace=0.55)

for panel_i, (title, indices, sublabels) in enumerate(GROUPS):
    ax = fig.add_subplot(gs[panel_i])
    for idx, lbl in zip(indices, sublabels):
        ax.plot(t, states[:, idx], label=lbl, linewidth=1.3)
    # Linea en 0: referencia visual para distinguir positivo/negativo
    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.7)
    # Limites fijos [-1.1, 1.1]: detectar si alguna variable se satura en +-1
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlim(t[0], t[-1])
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Valor norm.')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.8)
    ax.grid(True, alpha=0.25)

# Panel extra: distancia y score (no son entradas de la red, son referencias)
ax_dist  = fig.add_subplot(gs[n_panels - 1])
# twinx crea un segundo eje Y que comparte el eje X con ax_dist
ax_score = ax_dist.twinx()

ax_dist.plot(t, distances, color='royalblue', label='distancia (px)', linewidth=1.5)
ax_score.plot(t, scores, color='goldenrod', label='score', linewidth=1.5, linestyle='--')

ax_dist.set_title(
    'Metricas del episodio (referencia — no son entradas de la red)',
    fontsize=11, fontweight='bold',
)
ax_dist.set_xlabel('Tiempo (s)')
ax_dist.set_ylabel('Distancia (px)', color='royalblue')
ax_score.set_ylabel('Score', color='goldenrod')
lines1, labels1 = ax_dist.get_legend_handles_labels()
lines2, labels2 = ax_score.get_legend_handles_labels()
ax_dist.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
ax_dist.grid(True, alpha=0.25)
ax_dist.set_xlim(t[0], t[-1])

plt.savefig('state_episode.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada como experiments/state_episode.png')

In [ ]:
# sat% = % de frames donde |valor| >= 0.99 (el clamp de get_state() tuvo efecto).
# sat% > 10%: la constante de normalizacion es demasiado pequena para ese rango.

print("#".ljust(4) + "Variable".ljust(26) +
      "min".rjust(8) + "max".rjust(8) +
      "media".rjust(8) + "std".rjust(8) + "sat%".rjust(8))
print('-' * 72)

for i, label in enumerate(LABELS):
    col     = states[:, i]
    sat_pct = 100.0 * float(np.mean(np.abs(col) >= 0.99))
    flag    = '  <- revisar' if sat_pct > 10 else ''
    fmt1    = f'{i:<4}{label:<26}{col.min():>8.3f}{col.max():>8.3f}'
    fmt2    = f'{col.mean():>8.3f}{col.std():>8.3f}{sat_pct:>7.1f}%{flag}'
    print(fmt1 + fmt2)

print('-' * 72)
print('sat% > 10%: considerar aumentar la constante de normalizacion en environment.py')